In [1]:
import polars as pl
import polars.selectors as cs
import numpy as np

In [2]:
sushi_path = "./data/sushi3.idata"
sushi_df = pl.read_csv(
    sushi_path, separator="\t", has_header=False, 
    new_columns=[
        "id", "name", "style", "major_group",
        "minor_group", "oiliness", "eaten_frequency",
        "price", "sold_frequency"
    ]
)

sushi_df.head()

id,name,style,major_group,minor_group,oiliness,eaten_frequency,price,sold_frequency
i64,str,i64,i64,i64,f64,f64,f64,f64
0,"""ebi""",1,0,6,2.728978,2.138422,1.83842,0.84
1,"""anago""",1,0,3,0.926384,1.990228,1.992459,0.88
2,"""maguro""",1,0,1,1.769559,2.348506,1.874725,0.88
3,"""ika""",1,0,5,2.688401,2.04324,1.515152,0.92
4,"""uni""",1,0,8,0.813043,1.643478,3.287282,0.88


In [3]:
def prep_sushi(df, nominal_fields, numerical_fields, drop_fields, num_rows, id_field = "id"):
    df = df.limit(
        num_rows
    ).sort(
        id_field
    )
    arr = df.drop(
        drop_fields
    ).with_columns(
        (pl.col(numerical_fields) - pl.col(numerical_fields).min()) / (pl.col(numerical_fields).max() - pl.col(numerical_fields).min())
    ).with_columns(
        df[nominal_fields].to_dummies(drop_first=True)
    ).to_numpy()

    inds = (df[id_field].sort() + 1).to_list()

    return arr, inds

nominal_fields = ["minor_group"]
numerical_fields = ["oiliness", "eaten_frequency", "price", "sold_frequency"]
drop_fields = ["id", "name", "major_group"] + nominal_fields

sushi_arr, sushi_inds = prep_sushi(sushi_df, nominal_fields, numerical_fields, drop_fields, 10)

sushi_arr

array([[1.        , 0.85828263, 0.70201946, 0.2334073 , 0.5       ,
        0.        , 0.        , 0.        , 0.        , 0.        ,
        0.        ],
       [1.        , 0.14765004, 0.49182399, 0.2780176 , 0.75      ,
        0.        , 1.        , 0.        , 0.        , 0.        ,
        0.        ],
       [1.        , 0.48005296, 1.        , 0.24392127, 0.75      ,
        1.        , 0.        , 0.        , 0.        , 0.        ,
        0.        ],
       [1.        , 0.84228598, 0.56701463, 0.13978737, 1.        ,
        0.        , 0.        , 1.        , 0.        , 0.        ,
        0.        ],
       [1.        , 0.1029679 , 0.        , 0.65300401, 0.75      ,
        0.        , 0.        , 0.        , 0.        , 1.        ,
        0.        ],
       [1.        , 1.        , 0.10477308, 0.10190094, 0.        ,
        0.        , 0.        , 1.        , 0.        , 0.        ,
        0.        ],
       [1.        , 0.28109149, 0.47655331, 0.48158165, 0.

In [4]:
from mallows import TopKMallows, DiverseTopKMallows

In [265]:
algo_beta = 0.5
p_penalty = 0.321
algo_k = 2
universe = [0] + sushi_inds[:5]

algo_1 = TopKMallows(center=[1, 4, 3, 2, 5], k=algo_k, beta=algo_beta, p=p_penalty)

algo_1.profile_sampling_probs(universe)

defaultdict(<function mallows.TopKMallows.profile_sampling_probs.<locals>.<lambda>()>,
            {frozenset({1, 2}): np.float64(0.1413860199253529),
             frozenset({1, 3}): np.float64(0.1413860199253529),
             frozenset({1, 4}): np.float64(0.23310613843056144),
             frozenset({1, 5}): np.float64(0.1413860199253529),
             frozenset({2, 3}): np.float64(0.028490311324992083),
             frozenset({2, 4}): np.float64(0.08575495593946783),
             frozenset({2, 5}): np.float64(0.028490311324992083),
             frozenset({3, 4}): np.float64(0.08575495593946783),
             frozenset({3, 5}): np.float64(0.028490311324992083),
             frozenset({4, 5}): np.float64(0.08575495593946783)})

In [269]:
algo_alpha = 0.75
algo_sigma = 2.0

algo_2 = DiverseTopKMallows(center=[1, 4, 3, 2, 5], k=algo_k, beta=algo_beta, p=p_penalty, embeddings=sushi_arr, alpha=algo_alpha, sigma=algo_sigma)

algo_2.profile_sampling_probs(universe)

defaultdict(<function mallows.DiverseTopKMallows.profile_sampling_probs.<locals>.<lambda>()>,
            {frozenset({1, 2}): np.float64(0.14013440734199134),
             frozenset({1, 3}): np.float64(0.13733133630848632),
             frozenset({1, 4}): np.float64(0.18690531381082487),
             frozenset({1, 5}): np.float64(0.14546795188901032),
             frozenset({2, 3}): np.float64(0.03396748313567508),
             frozenset({2, 4}): np.float64(0.09568752390769833),
             frozenset({2, 5}): np.float64(0.03397138253834464),
             frozenset({3, 4}): np.float64(0.09524340567780817),
             frozenset({3, 5}): np.float64(0.03419147901644277),
             frozenset({4, 5}): np.float64(0.09709971637371839)})

## Using User Data

In [7]:
sushi_path = "./data/sushi3a.5000.10.order"
pref_arr = pl.read_csv(
    sushi_path, separator=" ", has_header=False, skip_lines=1
).select(
    cs.exclude(cs.by_index([0, 1]))
).to_numpy()

pref_list = pref_arr.tolist()
pref_arr

array([[5, 0, 3, ..., 1, 7, 2],
       [0, 9, 6, ..., 1, 5, 4],
       [7, 0, 2, ..., 1, 9, 6],
       ...,
       [7, 2, 4, ..., 0, 8, 9],
       [7, 2, 3, ..., 9, 6, 4],
       [0, 5, 3, ..., 9, 8, 1]], shape=(5000, 10))

In [8]:
items = [1, 2, 3, 4, 5]

for i in range(len(items)):
    for j in range(i+1, len(items)):
        print(items[i], items[j])

1 2
1 3
1 4
1 5
2 3
2 4
2 5
3 4
3 5
4 5


In [251]:
import pickle

"""
p_penalty = 0.321
algo_k = 10

furthest_ranks = {}

for i in range(len(pref_list)):
    a_tup = tuple(pref_list[i])
    for j in range(i+1, len(pref_list)):
        dist = TopKMallows.kendall_distance(pref_list[i], pref_list[j], algo_k, p_penalty)
        furthest_ranks[dist] = (pref_list[i], pref_list[j])
#"""

#"""
with open("data/pref_distances.pickle", 'rb') as file:
    furthest_ranks = pickle.load(file)
#"""

furthest_ranks

{21.0: ([7, 2, 3, 0, 8, 1, 5, 9, 6, 4], [0, 5, 3, 4, 7, 2, 6, 9, 8, 1]),
 25.0: ([6, 1, 9, 7, 4, 8, 3, 2, 0, 5], [7, 2, 4, 5, 3, 6, 1, 0, 8, 9]),
 18.0: ([7, 2, 4, 5, 3, 6, 1, 0, 8, 9], [0, 5, 3, 4, 7, 2, 6, 9, 8, 1]),
 16.0: ([0, 4, 5, 7, 8, 2, 6, 3, 1, 9], [7, 2, 4, 5, 3, 6, 1, 0, 8, 9]),
 23.0: ([2, 8, 3, 0, 5, 1, 6, 7, 9, 4], [6, 7, 0, 9, 2, 1, 8, 3, 5, 4]),
 27.0: ([7, 5, 1, 4, 0, 3, 8, 2, 9, 6], [6, 1, 9, 7, 4, 8, 3, 2, 0, 5]),
 26.0: ([7, 1, 3, 8, 6, 0, 5, 2, 9, 4], [0, 5, 3, 4, 7, 2, 6, 9, 8, 1]),
 22.0: ([7, 1, 3, 8, 6, 0, 5, 2, 9, 4], [7, 2, 4, 5, 3, 6, 1, 0, 8, 9]),
 32.0: ([7, 5, 2, 8, 0, 3, 9, 6, 1, 4], [6, 1, 9, 7, 4, 8, 3, 2, 0, 5]),
 20.0: ([0, 3, 6, 1, 7, 2, 8, 9, 5, 4], [0, 5, 3, 4, 7, 2, 6, 9, 8, 1]),
 9.0: ([1, 7, 8, 2, 0, 3, 6, 9, 5, 4], [7, 1, 3, 8, 6, 0, 5, 2, 9, 4]),
 28.0: ([6, 1, 9, 7, 4, 8, 3, 2, 0, 5], [7, 2, 3, 0, 8, 1, 5, 9, 6, 4]),
 15.0: ([0, 3, 6, 1, 7, 2, 8, 9, 5, 4], [7, 2, 3, 0, 8, 1, 5, 9, 6, 4]),
 29.0: ([6, 7, 0, 9, 2, 1, 8, 3, 5, 4], [7, 5, 1, 4,

In [10]:
"""
with open("data/pref_distances.pickle", 'wb') as file:
    pickle.dump(furthest_ranks, file)
#"""

'\nwith open("data/pref_distances.pickle", \'wb\') as file:\n    pickle.dump(furthest_ranks, file)\n#'

In [11]:
max(furthest_ranks)

45.0

In [300]:
dist = 5.0

furthest_ranks[dist]

([1, 7, 8, 2, 0, 3, 6, 9, 5, 4], [7, 2, 0, 1, 8, 3, 6, 9, 5, 4])

In [301]:
far_a = [ind + 1 for ind in furthest_ranks[dist][0]]
far_b = [ind + 1 for ind in furthest_ranks[dist][1]]

far_a

[2, 8, 9, 3, 1, 4, 7, 10, 6, 5]

In [302]:
far_b

[8, 3, 1, 2, 9, 4, 7, 10, 6, 5]

In [318]:
algo_beta = 1.0
human_beta = 1.0
p_penalty = 0.321
algo_k = 2
universe = [0] + sushi_inds

utility_arr = np.zeros(11)
utility_arr[far_a[0]] = 1
utility_arr

human = TopKMallows(center=far_a, k=10, beta=human_beta, p=p_penalty)

algo_1 = TopKMallows(center=far_b, k=algo_k, beta=algo_beta, p=p_penalty)

human.collab_utility(algo_1, universe, utility_arr)

np.float64(0.0901257889260058)

In [319]:
algo_alpha = 0.75
algo_sigma = 1.5
algo_2 = DiverseTopKMallows(center=far_b, k=algo_k, beta=algo_beta, p=p_penalty, embeddings=sushi_arr, alpha=algo_alpha, sigma=algo_sigma)

human.collab_utility(algo_2, universe, utility_arr)

np.float64(0.09288103299114468)